# 16_robustness_v2.ipynb

Task 5.11: chạy lại Thực nghiệm 6 và 7 trên bộ họ chưa thấy v2. Đo bốn hệ (không xử lý, oracle, bộ chọn, huấn luyện lại) rồi báo theo từng họ: macro F1, ΔF1 của nền, phần ΔF1 lấy lại được, phân phối lớp dự đoán, recall lớp ít ảnh kèm khoảng Wilson, và Spearman giữa MSD với mức hơn baseline.

Đặc tả, quy tắc kết luận và kỳ vọng E1 đến E10 đăng ký trước ở `docs/task-5.11-robustness-v2.md`. Tham số khóa ở `configs/robustness_v2.yaml`.

**Phạm vi.** Chỉ validation và chỉ họ vai trò train, val của bộ v2 (5 họ, 15 điều kiện). **Không mở tập test và không đo họ vai trò test**: những thứ đó để dành cho P6.1, mở một lần. Notebook dừng nếu `FINAL_EVAL` hoặc `FINAL_FAMILY_EVAL` đang bật.

**Không hiệu chỉnh lại bộ chọn.** Bộ chọn đã khóa ở 5.3 chạy nguyên trên reference v1 đóng băng. Kiểm tra phụ trên baseline mới chỉ để tham khảo.

## 1. Môi trường

Internet On, Accelerator GPU (T4). Input bắt buộc:

- `cthng123/durian-leaf-disease-dataset` (ảnh).
- Một dataset chứa checkpoint: `swin_tiny_seed42.pt` (reference 4.4) và 11 checkpoint của 5.9 và 5.10: `swin_tiny_ctrl_baseline_seed{42,43,44}.pt`, `swin_tiny_aug_seed{42,43,44}.pt`, `swin_tiny_trivialaugment_seed{42,43,44}.pt`, `swin_tiny_trivialaugment_geometric_seed42.pt`, `swin_tiny_aug_leave_group_seed42.pt`. Thiếu checkpoint nào thì notebook dừng và nêu tên.

Dựng ảnh suy giảm của các họ ngoài chạy trên CPU, có họ mất hơn nửa giây mỗi ảnh. Notebook dựng mỗi điều kiện đúng một lần rồi dùng lại cho mọi model.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "tn6-tn7-rerun"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Cấu hình và phạm vi

`load_settings` và `evaluation_conditions` từ chối họ vai trò test và tập test. In ra danh sách họ để kiểm bằng mắt trước khi tốn giờ GPU.

In [ ]:
import json
import os
import time
import zipfile

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

from src.corruption import load_seed
from src.evaluation import robustness_v2 as R
from src.evaluation.reference import collect_checkpoint, load_trained_model
from src.selector.calibrate import collect
from src.selector.pipeline import load_calibrated, load_selector_config
from src.training import run_control as C
from src.training.loop import resolve_device

for flag in ("FINAL_EVAL", "FINAL_FAMILY_EVAL"):
    assert os.environ.get(flag) != "1", f"{flag} đang bật. 5.11 không được mở tập test hay họ vai trò test."

settings = R.load_settings()
conditions = R.evaluation_conditions(settings)
names = list(conditions)
families = R.family_order(names)
seed = load_seed(settings["unseen_config"])
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
classes = list(pipeline_cfg["classes"])

REP = REPO_DIR / "reports/robustness_v2"
REP.mkdir(parents=True, exist_ok=True)

device = resolve_device()
if device.type != "cuda":
    raise RuntimeError("Notebook này cần GPU. Trong Session options chọn Accelerator GPU, rồi chạy lại từ cell đầu.")
WORKERS = max(1, min(8, os.cpu_count() or 1))
cv2.setNumThreads(1)

display(pd.DataFrame([{"họ": spec["family"], "vai trò": spec["role"], "nhóm MSD": spec["group"], "ba mức": [conditions[f"{spec['family']}_s{k}"]["value"] for k in (1, 2, 3)]}
                      for spec in {s["family"]: s for s in conditions.values()}.values()]))
print(f"{len(families)} họ, {len(names)} điều kiện, thiết bị {device}, seed ảnh suy giảm {seed}")

## 3. Dựng ảnh suy giảm một lần

`RetainedCache` giữ mọi ảnh đã dựng (khoảng 70 MB mỗi điều kiện), nên mọi hệ nhìn đúng cùng ảnh. Họ chậm nhất mất vài phút.

In [ ]:
from src.evaluation.robustness_v2 import RetainedCache

start = time.time()
cache = RetainedCache(pipeline_cfg, conditions, seed, split=settings["split"])
for name in ("clean", *names):
    cache.get(name)
    print(f"[{(time.time() - start) / 60:5.1f} phút] {name}", flush=True)
labels = cache.labels
print(f"{len(cache)} ảnh {settings['split']}, phân bố lớp: {dict(zip(classes, np.bincount(labels, minlength=len(classes))))}")

## 4. Reference đóng băng: không xử lý, bộ chọn, oracle

Bảng đặc trưng và dự đoán cho mọi hành động của bộ chọn (`collect`), rồi `system_predictions`: không xử lý, bộ chọn đã khóa ở 5.3 và ba cận oracle. Oracle dùng nhãn nên chỉ để so, cận theo ảnh bị thổi phồng.

In [ ]:
selector_cfg = load_selector_config(settings["selector"]["config"])
selector = load_calibrated(settings["selector"]["calibrated"], require_locked=True)
reference_cfg = yaml.safe_load((REPO_DIR / settings["reference"]["train_config"]).read_text(encoding="utf-8"))
reference_model = load_trained_model(
    reference_cfg, collect_checkpoint("/kaggle/input", settings["reference"]["checkpoint"], "/kaggle/working/checkpoints"), len(classes), device
)


def selector_table(model, label):
    start = time.time()
    table = collect(
        model, cache, conditions, selector_cfg["actions"], pipeline_cfg, device, workers=WORKERS,
        progress=lambda message: print(f"[{label} {(time.time() - start) / 60:5.1f} phút] {message}", flush=True),
    )
    assert table.conditions == ["clean", *names], "Thứ tự điều kiện của bảng khác danh sách điều kiện."
    assert np.array_equal(table.labels, labels), "Bảng và cache không cùng ảnh."
    return table


table_reference = selector_table(reference_model, "reference")
reference = R.system_predictions(table_reference, selector)
rates_reference = R.action_rates(table_reference, selector).assign(model="reference_v1")
display(rates_reference.drop(columns="model").round(3))
del reference_model
torch.cuda.empty_cache()

## 5. Kiểm tra phụ: cùng bộ chọn trên baseline mới

Bộ chọn khóa trên reference v1, nên nó chỉ là tham khảo trên baseline 5.9 seed 42. Không hiệu chỉnh lại. Mục đích: biết mức tăng có còn khi nền đổi.

In [ ]:
baseline_cfg, _ = C.load_arm_config(f"configs/controls/{settings['side_check']['arm']}.yaml")
side_stem = f"{baseline_cfg['model_name']}_seed{settings['side_check']['seed']}.pt"
side_model = load_trained_model(baseline_cfg, collect_checkpoint("/kaggle/input", side_stem, "/kaggle/working/checkpoints"), len(classes), device)
table_side = selector_table(side_model, "baseline mới")
side_check = R.system_predictions(table_side, selector)
rates_side = R.action_rates(table_side, selector).assign(model="baseline_5.9_seed42")
del side_model
torch.cuda.empty_cache()

## 6. Huấn luyện lại: nhánh của 5.9 và 5.10

Mỗi model chạy trên đúng ảnh trong cache, không xử lý ảnh trước khi nhận diện. Checkpoint tên `<model_name>_seed<seed>.pt`.

In [ ]:
plan = [(arm, s) for arm in R.THREE_SEED_ARMS for s in settings["retrained"]["seeds"]] + [(arm, 42) for arm in settings["retrained"]["single_seed"]]
assert set(settings["retrained"]["three_seed"]) == set(R.THREE_SEED_ARMS) and set(settings["retrained"]["single_seed"]) == set(R.SINGLE_SEED_ARMS)

retrained = {}
for arm, run_seed in plan:
    train_cfg, _ = C.load_arm_config(f"configs/controls/{arm}.yaml")
    stem = f"{train_cfg['model_name']}_seed{run_seed}.pt"
    model = load_trained_model(train_cfg, collect_checkpoint("/kaggle/input", stem, "/kaggle/working/checkpoints"), len(classes), device)
    retrained.setdefault(arm, {})[run_seed] = R.model_matrix(model, cache, names, pipeline_cfg, device)
    print(f"{arm} seed {run_seed}: macro F1 sạch {R.clean_score(labels, retrained[arm][run_seed], len(classes)):.4f}", flush=True)
    del model
    torch.cuda.empty_cache()

## 7. Bảng kết quả

Mọi hệ chấm trên cùng ảnh và cùng chỉ số bootstrap (1000 lần). Quy tắc kết luận ở `docs/task-5.11-robustness-v2.md`, mục 4: bộ chọn và oracle theo khoảng bootstrap ghép cặp, nhánh nhiều seed theo 2,5 lần SD seed.

In [ ]:
tables = R.build_tables(labels, names, classes, reference, retrained, settings, side_check)
tables["action_rates"] = pd.concat([rates_reference, rates_side], ignore_index=True)

wide = tables["condition_scores"].query("condition != 'clean'").groupby(["system", "family"])["macro_f1"].mean().unstack("family").loc[
    [*R.SYSTEMS, *R.THREE_SEED_ARMS, *R.SINGLE_SEED_ARMS]
]
print("Macro F1 trung bình ba mức, theo họ (nhánh nhiều seed gồm cả ba seed, nhìn bảng seed_spread để thấy SD):")
display(wide.round(4))

clean = tables["condition_scores"].query("condition == 'clean'").groupby("system")["macro_f1"].mean()
print("Macro F1 ảnh sạch:")
display(clean.round(4).to_frame("clean_f1"))

In [ ]:
print("Bộ chọn và oracle so với không xử lý (reference v1), khoảng bootstrap ghép cặp:")
display(tables["systems_vs_none"].round(4))
print("Kiểm tra phụ, cùng bộ chọn trên baseline mới seed 42 (tham khảo):")
display(tables["side_check_new_baseline"].round(4))

In [ ]:
print("Các cặp nhánh huấn luyện lại, quy tắc 2,5 lần SD seed:")
display(tables["retrained_comparison"].round(4))
print("Nhánh huấn luyện lại so với baseline, khoảng bootstrap theo ảnh (sai số lấy mẫu ảnh, không phải SD seed):")
display(tables["retrained_vs_baseline_ci"].round(4))
print("SD seed của từng nhánh ba seed:")
display(tables["seed_spread"].round(4))

In [ ]:
print("Phần ΔF1 của nền mà mỗi hệ lấy lại được (NaN khi nền gần như không giảm). Nền của bộ chọn và oracle là reference v1, nền của huấn luyện lại là baseline 5.9:")
display(tables["recovered_fraction"].round(4))

In [ ]:
diag = tables["diagnostics"]
print("Phân phối lớp dự đoán ở mức 3 (entropy thấp hơn nhãn thật nhiều là dấu hiệu dồn lớp):")
display(diag.query("severity == 3")[["system", "family", "macro_f1", "pred_entropy", "label_entropy", "top_class", "top_share"]].round(3))
print(f"Recall lớp {settings['statistics']['small_class']} ({int(diag['small_support'].iloc[0])} ảnh) ở mức 3, kèm khoảng Wilson 95%:")
display(diag.query("severity == 3")[["system", "family", "small_recall", "small_low", "small_high"]].round(3))

## 8. MSD và mức hơn baseline

Spearman giữa MSD tới TrivialAugment và mức TrivialAugment hơn baseline: 14 họ đã thấy (từ 5.9, trung bình 3 seed) cộng các họ v2 trong phạm vi. Spearman giữa MSD tới augmentation suy giảm và mức suy giảm hơn baseline: chỉ họ v2, vì 14 họ đã thấy nằm trong chính augmentation đó. Khoảng bootstrap rất rộng với ít họ, ghi vào kết luận.

In [ ]:
msd_record = json.loads((REPO_DIR / "reports/unseen_v2/msd.json").read_text(encoding="utf-8"))
vectors = R.msd_vectors(msd_record)

RES_5_9 = REPO_DIR / "reports/controls"
def seen_frames(arm):
    return [pd.read_csv(C.find_run(C.RunSpec(arm, s), RES_5_9)["conditions"]) for s in settings["retrained"]["seeds"]]

seen_ta = R.seen_effect(seen_frames("trivialaugment"), seen_frames("baseline"))
effects = tables["retrained_vs_baseline_ci"]
new_effects = {arm: {row.family: row.delta for row in effects[(effects["system"] == arm) & (effects["family"] != R.ALL_FAMILIES)].itertuples()} for arm in ("trivialaugment", "degradation")}
tables["msd_correlation"] = R.correlation_table(vectors, seen_ta, new_effects, families)
display(tables["msd_correlation"].round(3))

## 9. Lưu kết quả

Ghi bảng, dự đoán từng ảnh (int8) và ngữ cảnh chạy vào `reports/robustness_v2/`, rồi nén. Tải zip về, giải vào repo, commit, và đọc kết quả theo bảng E1 đến E10 ở `docs/task-5.11-robustness-v2.md`.

In [ ]:
for name, frame in tables.items():
    frame.to_csv(REP / f"{name}.csv", index=False)

arrays = {"labels": labels, "conditions": np.array(["clean", *names])}
arrays.update({f"reference__{name}": matrix.astype(np.int8) for name, matrix in reference.items()})
arrays.update({f"side_check__{name}": matrix.astype(np.int8) for name, matrix in side_check.items()})
arrays.update({f"retrained__{arm}__seed{s}": matrix.astype(np.int8) for arm, runs in retrained.items() for s, matrix in runs.items()})
np.savez_compressed(REP / "predictions_val.npz", **arrays)

context = {
    "task": "5.11", "split": settings["split"], "roles": settings["roles"], "families": families, "conditions": len(names), "images": int(len(labels)),
    "commit": commit, "branch": branch, "selector_sha256": json.loads((REPO_DIR / settings["selector"]["calibrated"]).read_text(encoding="utf-8"))["selector_sha256"],
    "unseen_config_sha256": yaml.safe_load((REPO_DIR / settings["unseen_config"]).read_text(encoding="utf-8"))["lock"]["sha256"],
    "statistics": settings["statistics"], "test_opened": False,
}
(REP / "run_context.json").write_text(json.dumps(context, ensure_ascii=False, indent=2), encoding="utf-8")

archive = Path("/kaggle/working/robustness_v2_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(REP.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive.name} ({archive.stat().st_size / 1024:.0f} KB). Tải về, giải vào repo (reports/robustness_v2/), rồi commit.")